# Last.fm Album Recommendations

Similar-album recommendations via Last.fm's `track.getSimilar` API.

Compare three ways of picking the seed track from the album:

- **(a) Top listener** — the track with the most Last.fm listeners on the album
- **(b) Random** — a random track from the album tracklist
- **(c) Top-N tracks** — use the `top_n` tracks (default 3) with the most listeners, aggregate recommendations by vote count; falls back to **(a)** if nothing is found

Set your API key in `bench/.env`:

```
LASTFM_API_KEY=your_key_here
```

In [1]:
import importlib

import pandas as pd

import lastfm_albums
importlib.reload(lastfm_albums)
from lastfm_albums import get_album_info, recommend_album

In [2]:
N_RECS = 1
FETCH_FLOOR = 10
TOP_N = 3
RANDOM_SEED = 42

## Seed album

In [3]:
SEED_ALBUM = "OK Computer"
SEED_ARTIST = "Radiohead"

In [4]:
seed = get_album_info(SEED_ARTIST, SEED_ALBUM)
pd.Series({k: seed[k] for k in seed if k != "tracks"})

key                                   radiohead::ok computer
artist                                             Radiohead
album                                            OK Computer
mbid                    0b6b4ba0-d36f-47bd-b4ea-6a5b91842d29
playcount                                          261022592
listeners                                            4835084
url          https://www.last.fm/music/Radiohead/OK+Computer
dtype: object

## Seed-track strategies

Run one cell at a time. Each calls `recommend_album()` directly.

In [5]:
# (a) Top-listener track
_, seed_track_a, recs_a, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_listener",
)
recs_a

,key,artist,album,url,similarity_score,seed_track,matched_via
0,pixies::surfer rosa,Pixies,Surfer Rosa,https://www.last.fm/music/Pixies/Surfer+Rosa,0.275238,No Surprises,track 'Where Is My Mind?' similar to 'No Surpr...


In [6]:
# (b) Random track
_, seed_track_b, recs_b, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="random",
    random_seed=RANDOM_SEED,
    clear_cache=False,
)
recs_b

,key,artist,album,url,similarity_score,seed_track,matched_via
0,jeff buckley::grace,Jeff Buckley,Grace,https://www.last.fm/music/Jeff+Buckley/Grace,0.162629,Lucky,track 'Grace' similar to 'Lucky'


In [7]:
# (c) Top-N tracks by listeners
_, seed_track_c, recs_c, used_fallback_c = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_n_tracks",
    top_n=TOP_N,
    clear_cache=False,
)
print(f"Needed to user fallback? {used_fallback_c}")
recs_c

Needed to user fallback? False


,key,artist,album,url,similarity_score,seed_track,matched_via,vote_count,seed_tracks
0,pixies::surfer rosa,Pixies,Surfer Rosa,https://www.last.fm/music/Pixies/Surfer+Rosa,0.326701,"No Surprises, Karma Police, Let Down","from 3 of 3 top tracks (No Surprises, Karma Po...",3,"[No Surprises, Karma Police, Let Down]"


In [8]:
# Overlap between (a) and (b)
recs_a.merge(recs_b, on="key", suffixes=("_top_listener", "_random"))

,key,artist_top_listener,album_top_listener,url_top_listener,similarity_score_top_listener,seed_track_top_listener,matched_via_top_listener,artist_random,album_random,url_random,similarity_score_random,seed_track_random,matched_via_random
